# ファクターモデル分解 — 残差相関行列による銘柄グループ構造分析

**実行環境**: Google Colab (無料枠) / Windows ローカル 自動判定  
**BQアクセス**: 初回1回のみ → ローカルキャッシュ。2回目以降はBQアクセスゼロ。

**データソース**:
- BQ `STOCK.STOCK_PRICE_JQUANTS` — 調整済み株価（ADJ_CLOSE）
- BQ `STOCK.STOCK_CODE_LIST` — 業種33・サイズ分類
- BQ `STOCK.fin_summary` — 発行済株式数（時価総額計算用）
- BQ `STOCK.INDEX_PRICE` — TOPIX（市場ファクター）
- 四季報 Excel — 連結事業・特色テキスト（クロス検証用、オプション）

**手法**:
1. 3ファクターモデル（市場・業種33・サイズ）で日次リターンを回帰
2. 残差の相関行列を年次ローリング（2020〜2025）で算出
3. 階層的クラスタリング＋PCAで銘柄グループ構造を可視化
4. 四季報テキスト類似度でクロス検証（オプション）

**出力**: Google Drive `analysis/factor_model/` に保存

In [ ]:
# ── Cell 1: pip install（Colab用）──────────────────────────────
%pip install -q japanize-matplotlib

In [ ]:
# ── Cell 2: Setup + Imports + 認証 ─────────────────────────────
%matplotlib inline
import os, sys, warnings
from pathlib import Path
from datetime import datetime, date

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
try:
    import japanize_matplotlib
except ImportError:
    pass
matplotlib.rcParams['axes.unicode_minus'] = False

from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from scipy.spatial.distance import squareform
from sklearn.decomposition import PCA
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

warnings.filterwarnings('ignore', category=FutureWarning)
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 100)

# ── 実行環境判定 ──
try:
    from google.colab import auth
    RUNTIME = 'colab'
except ImportError:
    RUNTIME = 'local'

# ── GCP認証 ──
from google.cloud import bigquery
if RUNTIME == 'colab':
    auth.authenticate_user()
    bq = bigquery.Client(project='gmailpj-357912')
else:
    from google.oauth2 import service_account
    import json
    _key_path = Path(r'C:\gdrive\claude\investment-agent\keys\gcp-service-account.json')
    _creds = service_account.Credentials.from_service_account_file(str(_key_path))
    bq = bigquery.Client(credentials=_creds, project='gmailpj-357912')

print(f'Runtime: {RUNTIME}')
print(f'BQ client ready: {bq.project}')

In [ ]:
# ── Cell 3: 設定パラメータ ──────────────────────────────────────
# ▼▼▼ ここを変更して使う ▼▼▼
YEAR_WINDOWS = [2020, 2021, 2022, 2023, 2024, 2025]  # 年次ローリング窓
MARKET_CAP_MIN = 50_000_000_000   # 時価総額下限: 500億円
MISSING_RATE_MAX = 0.05           # 欠損率上限: 5%
FORCE_RELOAD = False              # True=BQから強制再ダウンロード
N_CLUSTERS = 20                   # 階層クラスタリングのクラスタ数
N_PCA_COMPONENTS = 20             # PCA抽出主成分数
# ▲▲▲ ここを変更して使う ▲▲▲

# ── 四季報Excelパス（オプション） ──
# Colab: Google Driveにアップロード済みの場合
# ローカル: Dropboxパス
SHIKIHO_PATHS = [
    Path('/content/drive/MyDrive/stock/DA_四季報_2026_2.xlsx'),                 # Colab
    Path(r'C:\Users\zonekun\Dropbox\stock\DA_四季報_2026_2.xlsx'),  # Local
]

# ── キャッシュ・出力ディレクトリ ──
if RUNTIME == 'colab':
    CACHE_DIR = Path('/content/factor_model_cache')
    from google.colab import drive
    drive.mount('/content/drive')
    OUTPUT_DIR = Path('/content/drive/MyDrive/analysis/factor_model')
else:
    CACHE_DIR = Path(r'C:\tmp\factor_model_cache')
    OUTPUT_DIR = Path(r'C:\tmp\factor_model_output')

CACHE_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f'Cache: {CACHE_DIR}')
print(f'Output: {OUTPUT_DIR}')

## 1. データ取得（BQ → ローカルキャッシュ）

4テーブルを取得。初回のみBQアクセス、2回目以降はキャッシュ読み込み。

In [ ]:
# ── Cell 5: BQデータ取得 + キャッシュ ──────────────────────────
def cached_query(name: str, query: str) -> pd.DataFrame:
    """BQクエリ結果をCSVキャッシュする."""
    cache_path = CACHE_DIR / f'{name}.csv'
    if not FORCE_RELOAD and cache_path.exists():
        print(f'  {name}: キャッシュ読み込み')
        return pd.read_csv(cache_path)
    print(f'  {name}: BQからダウンロード...')
    df = bq.query(query).to_dataframe()
    df.to_csv(cache_path, index=False)
    return df

# --- 1. 株価（調整済み） ---
df_price_raw = cached_query('price_jquants', """
    SELECT TICKER, DATE, ADJ_CLOSE, VOLUME, TURNOVER
    FROM `gmailpj-357912.STOCK.STOCK_PRICE_JQUANTS`
    WHERE DATE >= '2019-12-01'
      AND ADJ_CLOSE IS NOT NULL
      AND VOLUME > 0
    ORDER BY TICKER, DATE
""")
df_price_raw['DATE'] = pd.to_datetime(df_price_raw['DATE'])
print(f'  → {len(df_price_raw):,} rows, {df_price_raw["TICKER"].nunique():,} tickers')

# --- 2. 銘柄マスタ ---
df_master = cached_query('stock_code_list', """
    SELECT TICKER, STOCK_NAME, MARKET_CATEGORY,
           INDUSTRY_33_CODE, INDUSTRY_33_CATEGORY,
           INDUSTRY_17_CODE, INDUSTRY_17_CATEGORY,
           SIZE_CODE, SIZE_CATEGORY
    FROM `gmailpj-357912.STOCK.STOCK_CODE_LIST`
    WHERE EXCHANGE = 'TSE'
""")
print(f'  → {len(df_master):,} tickers')

# --- 3. 発行済株式数（時価総額計算用） ---
df_shares_raw = cached_query('fin_summary_shares', """
    SELECT
        LOCAL_CODE AS TICKER,
        CURRENT_PERIOD_END_DATE AS PERIOD_END,
        NUMBER_OF_ISSUED_AND_OUTSTANDING_SHARES_AT_THE_END_OF_FISCAL_YEAR_INCLUDING_TREASURY_STOCK AS SHARES_ISSUED,
        NUMBER_OF_TREASURY_STOCK_AT_THE_END_OF_FISCAL_YEAR AS TREASURY_STOCK
    FROM `gmailpj-357912.STOCK.fin_summary`
    WHERE NUMBER_OF_ISSUED_AND_OUTSTANDING_SHARES_AT_THE_END_OF_FISCAL_YEAR_INCLUDING_TREASURY_STOCK IS NOT NULL
    ORDER BY LOCAL_CODE, CURRENT_PERIOD_END_DATE
""")
df_shares_raw['PERIOD_END'] = pd.to_datetime(df_shares_raw['PERIOD_END'])
print(f'  → {len(df_shares_raw):,} rows')

# --- 4. TOPIX（市場ファクター） ---
df_topix_raw = cached_query('topix', """
    SELECT DATE, CLOSE
    FROM `gmailpj-357912.STOCK.INDEX_PRICE`
    WHERE INDEX_CODE = '0000'
      AND DATE >= '2019-12-01'
    ORDER BY DATE
""")
df_topix_raw['DATE'] = pd.to_datetime(df_topix_raw['DATE'])
print(f'  → {len(df_topix_raw):,} rows')

print('\nデータ取得完了')

## 2. 時価総額フィルタ・欠損フィルタ

- 時価総額 500億円未満を除外
- 各年窓内で欠損率 5% 超を除外

In [ ]:
# ── Cell 7: 前処理 ─────────────────────────────────────────────

# --- 発行済株式数: 各年末時点の最新値を取得 ---
def get_shares_at_year_end(year: int) -> pd.DataFrame:
    """指定年末時点の直近の発行済株式数を取得."""
    cutoff = pd.Timestamp(f'{year}-12-31')
    df = df_shares_raw[df_shares_raw['PERIOD_END'] <= cutoff].copy()
    # 各TICKERの最新レコード
    idx = df.groupby('TICKER')['PERIOD_END'].idxmax()
    result = df.loc[idx, ['TICKER', 'SHARES_ISSUED', 'TREASURY_STOCK']].copy()
    result['TREASURY_STOCK'] = result['TREASURY_STOCK'].fillna(0)
    result['SHARES_OUTSTANDING'] = result['SHARES_ISSUED'] - result['TREASURY_STOCK']
    return result[['TICKER', 'SHARES_OUTSTANDING']]


# --- 日次対数リターン計算 ---
df_price_raw = df_price_raw.sort_values(['TICKER', 'DATE'])
df_price_raw['LOG_RETURN'] = df_price_raw.groupby('TICKER')['ADJ_CLOSE'].transform(
    lambda x: np.log(x / x.shift(1))
)

# --- TOPIXリターン ---
df_topix = df_topix_raw.sort_values('DATE').copy()
df_topix['MKT_RETURN'] = np.log(df_topix['CLOSE'] / df_topix['CLOSE'].shift(1))
df_topix = df_topix[['DATE', 'MKT_RETURN']].dropna()


def build_universe(year: int) -> tuple[pd.DataFrame, list[str]]:
    """指定年のユニバース（時価総額・欠損フィルタ適用済み）を構築.

    Returns:
        df_ret: ピボット済みリターン行列 (DATE × TICKER)
        tickers: 対象銘柄リスト
    """
    start = pd.Timestamp(f'{year}-01-01')
    end = pd.Timestamp(f'{year}-12-31')

    # 年内の株価データ
    mask = (df_price_raw['DATE'] >= start) & (df_price_raw['DATE'] <= end)
    df_year = df_price_raw[mask].copy()

    # 営業日数
    trading_days = df_year['DATE'].nunique()

    # --- 時価総額フィルタ ---
    # 年初の株価（最初の営業日）
    first_day = df_year.groupby('TICKER').first().reset_index()
    shares = get_shares_at_year_end(year - 1)  # 前年末の株式数
    mcap = first_day[['TICKER', 'ADJ_CLOSE']].merge(shares, on='TICKER', how='inner')
    mcap['MARKET_CAP'] = mcap['ADJ_CLOSE'] * mcap['SHARES_OUTSTANDING']
    large_tickers = set(mcap[mcap['MARKET_CAP'] >= MARKET_CAP_MIN]['TICKER'])

    # --- 欠損率フィルタ ---
    ticker_counts = df_year.groupby('TICKER')['DATE'].nunique()
    missing_rate = 1 - ticker_counts / trading_days
    low_missing_tickers = set(missing_rate[missing_rate <= MISSING_RATE_MAX].index)

    # --- マスタ結合（ETF・REIT除外） ---
    valid_master = df_master[
        df_master['INDUSTRY_33_CODE'].notna() &
        ~df_master['MARKET_CATEGORY'].isin(['ETF・ETN', 'REIT'])
    ]
    master_tickers = set(valid_master['TICKER'])

    # --- 最終ユニバース ---
    universe = large_tickers & low_missing_tickers & master_tickers
    tickers = sorted(universe)

    # ピボットテーブル: (DATE, TICKER) → LOG_RETURN
    df_univ = df_year[df_year['TICKER'].isin(universe)].drop_duplicates(
        subset=['DATE', 'TICKER'], keep='last'
    )
    df_ret = df_univ.pivot(
        index='DATE', columns='TICKER', values='LOG_RETURN'
    ).sort_index()
    # 最初の行は NaN（リターン計算不可）→ 除外
    df_ret = df_ret.iloc[1:]
    # 残りの欠損は 0 埋め（前日終値据え置き＝リターン0）
    df_ret = df_ret.fillna(0.0)

    print(f'  {year}: {len(tickers)} 銘柄 × {len(df_ret)} 日 '
          f'(除外: 時価総額 {len(master_tickers) - len(large_tickers & master_tickers)}, '
          f'欠損 {len(large_tickers & master_tickers) - len(universe)})')
    return df_ret, tickers


# 全年分のユニバース構築
universes = {}
for y in YEAR_WINDOWS:
    universes[y] = build_universe(y)

print(f'\nユニバース構築完了')

## 3. ファクター構築（MKT, IND33, SIZE）

- **MKT**: TOPIX 日次対数リターン
- **IND33**: 業種33別の等金額平均リターン（自銘柄除外なし — 銘柄数が多いので影響軽微）
- **SIZE**: サイズ区分別（Core30/Large70/Mid400/Small）の等金額平均リターン

In [ ]:
# ── Cell 9: ファクター構築 ──────────────────────────────────────

# マスタ情報を辞書化
ticker_to_ind33 = df_master.set_index('TICKER')['INDUSTRY_33_CODE'].to_dict()
ticker_to_size = df_master.set_index('TICKER')['SIZE_CODE'].to_dict()
ticker_to_name = df_master.set_index('TICKER')['STOCK_NAME'].to_dict()
ticker_to_ind33_name = df_master.set_index('TICKER')['INDUSTRY_33_CATEGORY'].to_dict()


def build_factors(year: int, df_ret: pd.DataFrame, tickers: list[str]) -> dict:
    """指定年のファクターリターン系列を構築.

    Returns:
        dict with keys: 'mkt', 'ind33' (dict of series), 'size' (dict of series)
    """
    start = pd.Timestamp(f'{year}-01-01')
    end = pd.Timestamp(f'{year}-12-31')

    # MKT: TOPIX
    mkt = df_topix[(df_topix['DATE'] >= start) & (df_topix['DATE'] <= end)].set_index('DATE')['MKT_RETURN']

    # IND33: 業種別等金額平均リターン
    ind33_map = {t: ticker_to_ind33.get(t) for t in tickers}
    ind33_factors = {}
    for code in set(ind33_map.values()):
        if code is None:
            continue
        members = [t for t, c in ind33_map.items() if c == code]
        if len(members) >= 2:
            ind33_factors[code] = df_ret[members].mean(axis=1)

    # SIZE: サイズ区分別等金額平均リターン
    size_map = {t: ticker_to_size.get(t) for t in tickers}
    size_factors = {}
    for code in set(size_map.values()):
        if code is None:
            continue
        members = [t for t, c in size_map.items() if c == code]
        if len(members) >= 2:
            size_factors[code] = df_ret[members].mean(axis=1)

    return {'mkt': mkt, 'ind33': ind33_factors, 'size': size_factors}


factors = {}
for y in YEAR_WINDOWS:
    df_ret, tickers = universes[y]
    factors[y] = build_factors(y, df_ret, tickers)
    n_ind = len(factors[y]['ind33'])
    n_size = len(factors[y]['size'])
    print(f'  {y}: MKT + {n_ind} IND33 + {n_size} SIZE ファクター')

print('\nファクター構築完了')

## 4. ローリング回帰 → 残差計算（年次6枚）

各銘柄ごとに OLS 回帰:  
`r_i(t) = α + β₁·MKT(t) + β₂·IND_j(t) + β₃·SIZE_k(t) + ε_i(t)`

同じ (industry, size) グループに属する銘柄はデザイン行列が同一 → バッチ回帰で高速化。

In [ ]:
# ── Cell 11: 回帰 → 残差 ────────────────────────────────────────

def compute_residuals(year: int) -> pd.DataFrame:
    """指定年の全銘柄について残差を計算.

    Returns:
        DataFrame: (DATE × TICKER) の残差行列
    """
    df_ret, tickers = universes[year]
    fac = factors[year]

    # 共通日付に揃える
    common_dates = df_ret.index.intersection(fac['mkt'].index)
    df_ret_aligned = df_ret.loc[common_dates]
    mkt_aligned = fac['mkt'].loc[common_dates].values

    T = len(common_dates)
    residuals = pd.DataFrame(index=common_dates, columns=tickers, dtype=float)

    # (industry, size) グループごとにバッチ回帰
    groups = {}
    for t in tickers:
        ind_code = ticker_to_ind33.get(t)
        size_code = ticker_to_size.get(t)
        key = (ind_code, size_code)
        groups.setdefault(key, []).append(t)

    for (ind_code, size_code), members in groups.items():
        # デザイン行列: [const, MKT, IND, SIZE]
        cols = [np.ones(T), mkt_aligned]

        if ind_code and ind_code in fac['ind33']:
            cols.append(fac['ind33'][ind_code].loc[common_dates].values)
        if size_code and size_code in fac['size']:
            cols.append(fac['size'][size_code].loc[common_dates].values)

        X = np.column_stack(cols)  # (T, K)
        Y = df_ret_aligned[members].values  # (T, N)

        # OLS: Beta = (X'X)^-1 X'Y
        try:
            beta = np.linalg.lstsq(X, Y, rcond=None)[0]  # (K, N)
            eps = Y - X @ beta  # (T, N)
        except np.linalg.LinAlgError:
            eps = Y  # フォールバック: 生リターンをそのまま使う

        residuals[members] = eps

    return residuals


# 全年分の残差計算
all_residuals = {}
for y in YEAR_WINDOWS:
    all_residuals[y] = compute_residuals(y)
    n_tickers = all_residuals[y].shape[1]
    n_days = all_residuals[y].shape[0]
    print(f'  {y}: 残差行列 {n_days} 日 × {n_tickers} 銘柄')

print('\n残差計算完了')

## 5. 残差相関行列 → 階層的クラスタリング → 可視化

- 残差のピアソン相関行列を計算
- Ward法で階層的クラスタリング → デンドログラム
- ヒートマップでクラスタ構造を可視化
- デンソー(6902)・アイシン(7259) の位置を確認

In [ ]:
# ── Cell 13: 相関行列 + クラスタリング + 可視化 ──────────────────

def analyze_and_plot(year: int, save: bool = True):
    """残差相関行列の計算・クラスタリング・可視化."""
    residuals = all_residuals[year]
    tickers = list(residuals.columns)

    # --- 相関行列 ---
    corr = residuals.corr()

    # --- 階層的クラスタリング（Ward法） ---
    # 相関距離: d = 1 - corr
    dist_matrix = 1 - corr.values
    np.fill_diagonal(dist_matrix, 0)
    # 数値誤差で微小な負値が出る場合の補正
    dist_matrix = np.maximum(dist_matrix, 0)
    condensed = squareform(dist_matrix)
    linkage_matrix = linkage(condensed, method='ward')
    cluster_labels = fcluster(linkage_matrix, t=N_CLUSTERS, criterion='maxclust')

    # クラスタラベルをDataFrameに
    df_clusters = pd.DataFrame({
        'TICKER': tickers,
        'CLUSTER': cluster_labels,
        'STOCK_NAME': [ticker_to_name.get(t, '') for t in tickers],
        'INDUSTRY_33': [ticker_to_ind33_name.get(t, '') for t in tickers],
    }).sort_values(['CLUSTER', 'TICKER'])

    # --- デンドログラム ---
    fig, axes = plt.subplots(2, 1, figsize=(20, 16))

    # 上段: デンドログラム（上位30クラスタまで圧縮表示）
    ax = axes[0]
    dendrogram(
        linkage_matrix,
        truncate_mode='lastp',
        p=30,
        ax=ax,
        leaf_font_size=8,
    )
    ax.set_title(f'{year} 残差相関デンドログラム（Ward法, {len(tickers)}銘柄）', fontsize=14)
    ax.set_ylabel('距離')

    # 下段: クラスタ順に並べ替えた相関ヒートマップ
    ax = axes[1]
    order = np.argsort(cluster_labels)
    corr_ordered = corr.values[np.ix_(order, order)]
    im = ax.imshow(corr_ordered, cmap='RdBu_r', vmin=-0.3, vmax=0.3, aspect='auto')
    ax.set_title(f'{year} 残差相関ヒートマップ（クラスタ順）', fontsize=14)
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

    plt.tight_layout()
    if save:
        fig.savefig(OUTPUT_DIR / f'residual_corr_{year}.png', dpi=150, bbox_inches='tight')
    plt.show()

    # --- デンソー・アイシンの確認 ---
    spotlight = ['6902', '7259']  # デンソー, アイシン
    for t in spotlight:
        if t in tickers:
            cl = df_clusters[df_clusters['TICKER'] == t]['CLUSTER'].values[0]
            name = ticker_to_name.get(t, '')
            same_cluster = df_clusters[df_clusters['CLUSTER'] == cl]
            print(f'  {t} ({name}): クラスタ {cl} ({len(same_cluster)}銘柄)')
            print(f'    同クラスタ: {list(same_cluster["STOCK_NAME"].values[:10])}')

    return corr, df_clusters, linkage_matrix


# 全年分を実行
all_corr = {}
all_clusters = {}
all_linkages = {}

for y in YEAR_WINDOWS:
    print(f'\n=== {y} ===')
    corr, clusters, link = analyze_and_plot(y)
    all_corr[y] = corr
    all_clusters[y] = clusters
    all_linkages[y] = link

## 6. PCA on 残差 → 隠れグループ構造

残差相関行列にPCAをかけ、業種・サイズでは説明できない隠れたファクター構造を発見する。

In [ ]:
# ── Cell 15: PCA ────────────────────────────────────────────────

def residual_pca(year: int, save: bool = True):
    """残差にPCAをかけ、主成分ローディングを分析."""
    residuals = all_residuals[year]
    tickers = list(residuals.columns)
    n_comp = min(N_PCA_COMPONENTS, len(tickers) - 1)

    pca = PCA(n_components=n_comp)
    pca.fit(residuals.values)  # (T, N) → 主成分は銘柄方向

    # --- 寄与率プロット ---
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))

    ax = axes[0]
    ax.bar(range(1, n_comp + 1), pca.explained_variance_ratio_ * 100)
    ax.set_xlabel('主成分')
    ax.set_ylabel('寄与率 (%)')
    ax.set_title(f'{year} 残差PCA 寄与率')

    ax = axes[1]
    cumulative = np.cumsum(pca.explained_variance_ratio_) * 100
    ax.plot(range(1, n_comp + 1), cumulative, 'o-')
    ax.set_xlabel('主成分数')
    ax.set_ylabel('累積寄与率 (%)')
    ax.set_title(f'{year} 残差PCA 累積寄与率')
    ax.axhline(y=50, color='r', linestyle='--', alpha=0.5)

    plt.tight_layout()
    if save:
        fig.savefig(OUTPUT_DIR / f'pca_variance_{year}.png', dpi=150, bbox_inches='tight')
    plt.show()

    # --- 上位5主成分のトップ/ボトム銘柄 ---
    loadings = pd.DataFrame(
        pca.components_.T,  # (N, n_comp)
        index=tickers,
        columns=[f'PC{i+1}' for i in range(n_comp)]
    )

    print(f'\n{year}: 上位5主成分の特徴的な銘柄')
    for pc_idx in range(min(5, n_comp)):
        pc_name = f'PC{pc_idx + 1}'
        var_pct = pca.explained_variance_ratio_[pc_idx] * 100
        top5 = loadings[pc_name].nlargest(5)
        bottom5 = loadings[pc_name].nsmallest(5)
        print(f'\n  {pc_name} (寄与率 {var_pct:.1f}%):')
        print(f'    +: {[", ".join(f"{t}({ticker_to_name.get(t, "")})" for t in top5.index)]}')
        print(f'    -: {[", ".join(f"{t}({ticker_to_name.get(t, "")})" for t in bottom5.index)]}')

    return loadings, pca


all_loadings = {}
all_pca = {}

for y in YEAR_WINDOWS:
    print(f'\n{"="*60}')
    loadings, pca = residual_pca(y)
    all_loadings[y] = loadings
    all_pca[y] = pca

## 7. 四季報テキスト類似度 → クロス検証（オプション）

四季報の「連結事業」「特色」テキストから事業類似度を算出し、残差相関と比較する。

- **残差相関 高 × テキスト類似 高** → 事業が似ていて株価も連動（期待通り）
- **残差相関 高 × テキスト類似 低** → サプライチェーン・株主構造等の隠れた関係
- **残差相関 低 × テキスト類似 高** → 市場が区別しているペア（差別化が効いている）

In [ ]:
# ── Cell 17: 四季報テキスト類似度 ──────────────────────────────

# 四季報Excelを探す
shikiho_path = None
for p in SHIKIHO_PATHS:
    if p.exists():
        shikiho_path = p
        break

if shikiho_path is None:
    print('四季報Excelが見つかりません。このセクションをスキップします。')
    print('Google Drive に四季報Excelをアップロードしてパスを設定してください。')
    df_shikiho = None
else:
    print(f'四季報読み込み: {shikiho_path}')
    df_shikiho = pd.read_excel(shikiho_path, sheet_name='list')
    # コード正規化
    df_shikiho['CODE'] = df_shikiho['コード'].astype(str).str.zfill(4)
    # テキスト結合: 連結事業 + 特色
    text_cols = ['連結事業', '特色']
    available_cols = [c for c in text_cols if c in df_shikiho.columns]
    df_shikiho['TEXT'] = df_shikiho[available_cols].fillna('').astype(str).agg(' '.join, axis=1)
    df_shikiho = df_shikiho[df_shikiho['TEXT'].str.strip().str.len() > 0]
    print(f'  テキスト付き銘柄数: {len(df_shikiho)}')

In [ ]:
# ── Cell 18: テキスト類似度 × 残差相関のクロス検証 ──────────────

if df_shikiho is not None:
    # 直近年の残差相関を使う
    latest_year = max(YEAR_WINDOWS)
    corr_latest = all_corr[latest_year]
    tickers_latest = list(corr_latest.columns)

    # 四季報テキストがある銘柄に絞る
    shikiho_codes = set(df_shikiho['CODE'])
    common_tickers = [t for t in tickers_latest if t in shikiho_codes]
    print(f'残差相関 ∩ 四季報: {len(common_tickers)} 銘柄')

    # TF-IDF（文字n-gram: 日本語トークナイザ不要）
    shikiho_indexed = df_shikiho.set_index('CODE')
    texts = [shikiho_indexed.loc[t, 'TEXT'] for t in common_tickers]

    tfidf = TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 4), max_features=5000)
    tfidf_matrix = tfidf.fit_transform(texts)
    text_sim = cosine_similarity(tfidf_matrix)  # (N, N)

    # 残差相関（共通銘柄のみ）
    resid_corr = corr_latest.loc[common_tickers, common_tickers].values

    # 上三角のみ取得（対角除く）
    N = len(common_tickers)
    iu = np.triu_indices(N, k=1)
    resid_flat = resid_corr[iu]
    text_flat = text_sim[iu]

    # --- 散布図 ---
    fig, ax = plt.subplots(figsize=(10, 8))
    ax.scatter(text_flat, resid_flat, alpha=0.02, s=1)
    ax.set_xlabel('テキスト類似度（TF-IDF コサイン）')
    ax.set_ylabel('残差相関')
    ax.set_title(f'{latest_year} テキスト類似度 vs 残差相関 ({N}銘柄, {len(resid_flat):,}ペア)')
    ax.axhline(y=0, color='gray', linewidth=0.5)
    ax.axvline(x=0.3, color='red', linewidth=0.5, linestyle='--', alpha=0.5)
    ax.axhline(y=0.1, color='red', linewidth=0.5, linestyle='--', alpha=0.5)
    plt.tight_layout()
    fig.savefig(OUTPUT_DIR / f'text_vs_residcorr_{latest_year}.png', dpi=150, bbox_inches='tight')
    plt.show()

    # --- 注目ペア抽出 ---
    # 残差相関 > 0.15 かつ テキスト類似 < 0.1 → 隠れた関係
    hidden_mask = (resid_flat > 0.15) & (text_flat < 0.1)
    hidden_pairs = [(common_tickers[iu[0][i]], common_tickers[iu[1][i]],
                     resid_flat[i], text_flat[i])
                    for i in np.where(hidden_mask)[0]]
    hidden_pairs.sort(key=lambda x: -x[2])  # 残差相関降順

    print(f'\n隠れた関係（残差相関>0.15, テキスト類似<0.1）: {len(hidden_pairs)} ペア')
    print('Top 20:')
    for t1, t2, rc, ts in hidden_pairs[:20]:
        n1 = ticker_to_name.get(t1, '')
        n2 = ticker_to_name.get(t2, '')
        print(f'  {t1}({n1}) × {t2}({n2}): 残差相関={rc:.3f}, テキスト類似={ts:.3f}')

    # 全体相関（テキスト類似度と残差相関の相関）
    overall_corr = np.corrcoef(text_flat, resid_flat)[0, 1]
    print(f'\nテキスト類似度 × 残差相関 の相関係数: {overall_corr:.4f}')
else:
    print('四季報データなし。スキップ。')

## 8. 出力保存（Google Drive）

他モデルの入力として使える形式で保存:
- 残差相関行列 (年×CSV)
- ファクターβ係数
- PCAローディング
- クラスタラベル

In [ ]:
# ── Cell 20: 出力保存 ──────────────────────────────────────────

for y in YEAR_WINDOWS:
    # 残差相関行列
    all_corr[y].to_csv(OUTPUT_DIR / f'residual_corr_{y}.csv')

    # クラスタラベル
    all_clusters[y].to_csv(OUTPUT_DIR / f'clusters_{y}.csv', index=False)

    # PCAローディング
    loadings = all_loadings[y].copy()
    loadings['STOCK_NAME'] = [ticker_to_name.get(t, '') for t in loadings.index]
    loadings['INDUSTRY_33'] = [ticker_to_ind33_name.get(t, '') for t in loadings.index]
    loadings.to_csv(OUTPUT_DIR / f'pca_loadings_{y}.csv')

    print(f'  {y}: 保存完了')

# ファクターβ係数（全年まとめ）
beta_records = []
for y in YEAR_WINDOWS:
    residuals = all_residuals[y]
    df_ret, tickers = universes[y]
    fac = factors[y]
    common_dates = df_ret.index.intersection(fac['mkt'].index)
    mkt_aligned = fac['mkt'].loc[common_dates].values

    for t in tickers:
        ind_code = ticker_to_ind33.get(t)
        size_code = ticker_to_size.get(t)

        cols = [np.ones(len(common_dates)), mkt_aligned]
        col_names = ['alpha', 'beta_mkt']
        if ind_code and ind_code in fac['ind33']:
            cols.append(fac['ind33'][ind_code].loc[common_dates].values)
            col_names.append('beta_ind33')
        if size_code and size_code in fac['size']:
            cols.append(fac['size'][size_code].loc[common_dates].values)
            col_names.append('beta_size')

        X = np.column_stack(cols)
        y_vals = df_ret.loc[common_dates, t].values
        try:
            beta = np.linalg.lstsq(X, y_vals, rcond=None)[0]
        except np.linalg.LinAlgError:
            beta = [np.nan] * len(col_names)

        record = {'YEAR': y, 'TICKER': t}
        for name, val in zip(col_names, beta):
            record[name] = val
        beta_records.append(record)

df_betas = pd.DataFrame(beta_records)
df_betas['STOCK_NAME'] = df_betas['TICKER'].map(ticker_to_name)
df_betas['INDUSTRY_33'] = df_betas['TICKER'].map(ticker_to_ind33_name)
df_betas.to_csv(OUTPUT_DIR / 'factor_betas_all_years.csv', index=False)
print(f'\nファクターβ係数: {len(df_betas)} レコード → factor_betas_all_years.csv')

# サマリー統計
print(f'\n=== 出力先: {OUTPUT_DIR} ===')
for f in sorted(OUTPUT_DIR.glob('*')):
    size_kb = f.stat().st_size / 1024
    print(f'  {f.name}: {size_kb:.0f} KB')

## 9. サマリー・次ステップ

### 出力ファイル一覧

| ファイル | 内容 | 用途 |
|---------|------|------|
| `residual_corr_YYYY.csv` | N×N 残差相関行列 | 銘柄間の固有連動度 |
| `clusters_YYYY.csv` | 銘柄・クラスタID・業種 | グループ分類ラベル |
| `pca_loadings_YYYY.csv` | 銘柄×PC ローディング | 隠れグループの所属度 |
| `factor_betas_all_years.csv` | 銘柄×年×β | ファクター感応度の時系列変化 |

### 他モデルでの利用例

- **リスクモデル**: 残差相関行列でポートフォリオ分散を推定
- **ペアトレード候補**: 同クラスタ・高残差相関ペアをスクリーニング
- **セクターローテーション**: PCAローディングの時系列変化でテーマシフトを検出
- **イベントスタディ**: 決算発表後、同クラスタ銘柄への波及効果を定量化